# Fine-tune pretrained YOLO11n on CUDA
First run notebook 01 if the dataset has not been prepared. It is already prepared in this workspace.

**To start training yourself:** open this notebook and choose Run All, or run its three code cells in order (setup, progress display, training). The final cell starts a fresh 20-epoch run. The assistant has not executed the new training cell.

Expect roughly **20–25 minutes** on the RTX 5060 Ti at the current settings; early stopping can finish sooner. This estimate comes from the interrupted initial run, not a completed benchmark. Startup, validation, plotting and disk speed affect it.

The live display shows overall percentage, elapsed time and estimated time remaining. The ETA becomes available after the first complete epoch and includes observed validation overhead. During training it is an estimate based on epochs and batches, not a guarantee. Completion stays below 100% until checkpoint finalization finishes. Early stopping marks the run complete and shows how many epochs ran. Total duration is printed and saved to latest_run.json.

YOLO11n uses pretrained COCO features with a new two-class head for ID detection. Documentation: https://docs.ultralytics.com/models/yolo11/ and https://docs.ultralytics.com/modes/train/

20 epochs, 640px, batch 16, explicit CUDA device 0, workers=0. Lower batch to 8 on GPU memory errors. Each run uses a unique output directory. Validation selects the checkpoint; test is reserved for notebook 03. Evaluate real phone captures before relying on this model for delivery.


In [4]:
from pathlib import Path
from datetime import datetime
import json
import sys
import torch
import ultralytics
from ultralytics import YOLO

PROJECT = Path('/home/joe/PycharmProjects/id-capture')
DATA = PROJECT / 'training/data/midv500_yolo'
assert (DATA / 'data.yaml').is_file(), 'Run notebook 01 first.'
assert torch.cuda.is_available(), 'CUDA is required; no silent CPU fallback.'
probe = torch.randn(128, 128, device='cuda')
assert torch.isfinite(probe @ probe.T).all().item()
torch.cuda.synchronize()
del probe
print('Python:', sys.executable)
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('Training GPU:', torch.cuda.get_device_name(0))
RUN_NAME = 'yolo11n_midv500_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUNS = PROJECT / 'training/runs'
EPOCHS = 20
BATCH = 16
IMGSZ = 640
WEIGHTS = PROJECT / 'models/yolo11n.pt'
if not WEIGHTS.exists():
    from ultralytics.utils.downloads import attempt_download_asset
    downloaded = attempt_download_asset(str(WEIGHTS))
    assert Path(downloaded).is_file(), downloaded
model = YOLO(str(WEIGHTS))
print('Pretrained checkpoint:', WEIGHTS)
print('Run:', RUN_NAME)


Python: /home/joe/.cache/pypoetry/virtualenvs/id-capture-MVkOamB--py3.12/bin/python3.12
Ultralytics: 8.4.175
PyTorch: 2.14.1+cu130 CUDA: 13.0
Training GPU: NVIDIA GeForce RTX 5060 Ti
Pretrained checkpoint: /home/joe/PycharmProjects/id-capture/models/yolo11n.pt
Run: yolo11n_midv500_20261012_095811


In [5]:
import time
import html
from datetime import timedelta
import ipywidgets as widgets
from IPython.display import display

def duration(seconds):
    return str(timedelta(seconds=int(max(0, seconds))))

class TrainingProgress:
    def __init__(self, epochs):
        self.epochs = epochs
        self.started = time.perf_counter()
        self.last_update = 0.0
        self.last_print = 0.0
        self.batch = 0
        self.completed_epochs = 0
        self.epoch_seconds = None
        self.bar = widgets.FloatProgress(value=0, min=0, max=100, description='Completion:', bar_style='info')
        self.details = widgets.HTML()
        display(widgets.VBox([self.bar, self.details]))
        self.update(0, 'Preparing training', force=True)

    def update(self, completed, phase, force=False):
        now = time.perf_counter()
        if not force and now - self.last_update < 1:
            return
        self.last_update = now
        elapsed = now - self.started
        percentage = min(99.9, 100 * completed / self.epochs)
        self.bar.value = percentage
        remaining = 'estimating after first epoch'
        if self.epoch_seconds is not None:
            remaining = duration((self.epochs - completed) * self.epoch_seconds)
        self.details.value = (
            f'<b>{percentage:.1f}%</b> | {html.escape(phase)} | '
            f'Elapsed: {duration(elapsed)} | Estimated remaining: {remaining}'
        )

        # Text updates also work in notebook viewers without widget rendering.
        if force or now - self.last_print >= 10:
            self.last_print = now
            print(f'{percentage:.1f}% | {phase} | Elapsed: {duration(elapsed)} | Estimated remaining: {remaining}', flush=True)

    def epoch_start(self, trainer):
        self.batch = 0
        self.update(trainer.epoch, f'Epoch {trainer.epoch + 1}/{trainer.epochs}', force=True)

    def batch_end(self, trainer):
        self.batch += 1
        batches = max(1, len(trainer.train_loader))
        completed = trainer.epoch + min(1, self.batch / batches)
        self.update(completed, f'Epoch {trainer.epoch + 1}/{trainer.epochs}, batch {self.batch}/{batches}')

    def epoch_end(self, trainer):
        completed = trainer.epoch + 1
        if completed > trainer.epochs or completed <= self.completed_epochs:
            return  # Final validation repeats this callback; it is not another epoch.
        self.completed_epochs = max(self.completed_epochs, completed)
        self.epoch_seconds = (time.perf_counter() - self.started) / max(1, self.completed_epochs)
        self.update(completed, f'Completed {self.completed_epochs}/{trainer.epochs} epochs', force=True)

    def finish(self):
        elapsed = time.perf_counter() - self.started
        self.bar.value = 100
        self.bar.bar_style = 'success'
        reason = ' (early stopping)' if self.completed_epochs < self.epochs else ''
        self.details.value = f'<b>100% — run complete{reason}</b> | {self.completed_epochs}/{self.epochs} epochs | Total time: {duration(elapsed)}'
        print(f'Training run complete{reason}. Total time: {duration(elapsed)}.')

    def failed(self, interrupted=False):
        self.bar.bar_style = 'warning' if interrupted else 'danger'
        self.details.value = (
            f'<b>{"Interrupted" if interrupted else "Failed"} at {self.bar.value:.1f}%</b> | '
            f'Elapsed: {duration(time.perf_counter() - self.started)}'
        )

In [6]:
# Start a fresh run whenever this cell is executed.
RUN_NAME = 'yolo11n_midv500_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
model = YOLO(str(WEIGHTS))
progress = TrainingProgress(EPOCHS)
model.add_callback('on_train_epoch_start', progress.epoch_start)
model.add_callback('on_train_batch_end', progress.batch_end)
model.add_callback('on_fit_epoch_end', progress.epoch_end)

try:
    train_results = model.train(
        data=str(DATA / 'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        device=0, workers=0, seed=42, deterministic=True, amp=True,
        project=str(RUNS), name=RUN_NAME, exist_ok=False, patience=8,
        optimizer='auto', cache=False, plots=True, save=True,
        degrees=10.0, fliplr=0.0, flipud=0.0, close_mosaic=5,
    )
    assert model.trainer.device.type == 'cuda', 'The trainer did not use CUDA.'
    BEST = Path(model.trainer.best)
    assert BEST.is_file(), BEST
    total_seconds = time.perf_counter() - progress.started
    pointer = {
        'best': str(BEST), 'run_dir': str(BEST.parent.parent),
        'dataset': str(DATA / 'data.yaml'), 'epochs_requested': EPOCHS,
        'epochs_completed': progress.completed_epochs, 'imgsz': IMGSZ,
        'total_seconds': total_seconds,
    }
    (DATA / 'latest_run.json').write_text(json.dumps(pointer, indent=2))
    progress.finish()
    print('PASS: trained on', model.trainer.device)
    print('Best checkpoint:', BEST)
    print('Next: open notebook 03 for held-out test evaluation and inference.')
except KeyboardInterrupt:
    progress.failed(interrupted=True)
    raise
except Exception:
    progress.failed()
    raise

0.0% | Preparing training | Elapsed: 0:00:00 | Estimated remaining: estimating after first epoch
Ultralytics 8.4.175 🚀 Python-3.12.13 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 15850MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/joe/PycharmProjects/id-capture/training/data/midv500_yolo/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

In file included from /home/joe/.pyenv/versions/3.12.13/include/python3.12/Python.h:12,
                 from /home/joe/.cache/pypoetry/virtualenvs/id-capture-MVkOamB--py3.12/lib/python3.12/site-packages/triton/backends/nvidia/driver.c:9:
/home/joe/.pyenv/versions/3.12.13/include/python3.12/pyconfig.h:1874:9: warning: ‘_POSIX_C_SOURCE’ redefined
 1874 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/x86_64-linux-gnu/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/joe/.cache/pypoetry/virtualenvs/id-capture-MVkOamB--py3.12/lib/python3.12/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /home/joe/.cache/pypoetry/virtualenvs/id-capture-MVkOamB--py3.12/lib/python3.12/site-packages/triton/backends/nvidia/driver.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      | 

YOLO11n summary (fused): 101 layers, 2,597,294 parameters, 0 gradients, 6.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 4.5it/s 6.9s0.2s
                   all        985        495      0.982      0.968      0.991      0.973
       driving_licence        199        199      0.965       0.96       0.99      0.973
              passport        296        296          1      0.976      0.993      0.973
Speed: 0.1ms preprocess, 0.7ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /home/joe/PycharmProjects/id-capture/training/runs/yolo11n_midv500_20261012_095837_074736
99.9% | Completed 21/20 epochs | Elapsed: 0:17:26 | Estimated remaining: 0:00:00


AssertionError: 

## Recover the completed run without retraining
Your 20-epoch run finished successfully, but an incorrect post-training assertion prevented latest_run.json from being written. Run ONLY the recovery cell below for that existing run; do not choose Run All or run the training cell again. Then open notebook 03 and run its code cells in order.
The observed duration was approximately 17 minutes 26 seconds, including finalization. The recovery cell reads results.csv to count actual training epochs and writes the checkpoint pointer. It does not run training or inference.


In [7]:
# RECOVER SAVED MODEL — does not retrain.
from pathlib import Path
import csv
import json

PROJECT = Path('/home/joe/PycharmProjects/id-capture')
DATA = PROJECT / 'training/data/midv500_yolo'
RUN_DIR = PROJECT / 'training/runs/yolo11n_midv500_20261012_095837_074736'
BEST = RUN_DIR / 'weights/best.pt'
assert BEST.is_file(), BEST
with (RUN_DIR / 'results.csv').open(newline='') as handle:
    rows = list(csv.DictReader(handle))
completed = max(int(float(row['epoch'])) for row in rows)
pointer = {
    'best': str(BEST), 'run_dir': str(RUN_DIR),
    'dataset': str(DATA / 'data.yaml'), 'epochs_requested': 20,
    'epochs_completed': completed, 'imgsz': 640,
}
reference = DATA / 'latest_run.json'
if reference.exists():
    pointer = json.loads(reference.read_text())
    print('Preserving existing checkpoint reference:', pointer['best'])
else:
    reference.write_text(json.dumps(pointer, indent=2))
print('Saved checkpoint reference:', BEST)
print('Training epochs completed:', completed)
print('Ready: open notebook 03 and run its code cells in order.')

Saved checkpoint reference: /home/joe/PycharmProjects/id-capture/training/runs/yolo11n_midv500_20261012_095837_074736/weights/best.pt
Training epochs completed: 20
Ready: open notebook 03 and run its code cells in order.
